# Imports moduls


In [1]:
import tensorflow as tf 
from tensorflow.python.client import device_lib
import numpy as np
#import pandas as pd
import math
import os
from tensorflow import keras
from keras import backend as K
#import graphviz
from tensorflow.keras.layers import Dense, Activation, Permute, Dropout, Reshape
from scipy.io import loadmat
from tensorflow.keras.layers import BatchNormalization

# Important to load the data
import scipy.io
import glob
import os
import pandas as pd
from pathlib import Path
import pickle
from tensorboard import notebook

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from collections import Counter
import collections
#from imblearn.over_sampling import RandomOverSampler
#from imblearn.under_sampling import ClusterCentroids, RandomUnderSampler

# Imports for the construction of the CNN
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.layers import Flatten, GaussianNoise, Dropout
from keras.layers import Input, Dense, concatenate, LSTM, Masking, Conv2D, MaxPooling1D, Conv1D, ConvLSTM2D, Dropout
from keras.layers import MaxPooling2D, GlobalMaxPooling1D, Add, MaxPool2D, TimeDistributed, MaxPool3D, Input
from keras.models import Model
from keras.models import Sequential
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.utils import plot_model
from keras.utils import to_categorical

# Used to save the model if we desire to test anything else with it
import datetime as dt
import time as time

# Used to plot graphs for the evaluation of the models
from matplotlib import pyplot as plt
import seaborn as sns
# Used to evaluate the mohttp://localhost:8888/notebooks/DatiTesiRAW/PyPNRelay/main.ipynb#dels we construct
from sklearn import metrics
from sklearn.metrics import confusion_matrix

import absl.logging
import datetime

# Load the TensorBoard notebook extension
%load_ext tensorboard

absl.logging.set_verbosity(absl.logging.ERROR)

# Imports toolbox modules


In [2]:
from toolbox.models.EEGModels import EEGNet, EEGNet4x, EEGNetK150, EEGNetK50, EEGNetNoConv2D, EEGNetNoDepthConv2D
from toolbox.models.CNNModels import CNN_temp_elec, CONV_LSTM, CNN_4x, CNN_4xKL_v2
from toolbox.functions.Functions import get_file_paths, import_sample, get_label_from_path, encod_label, encod_label_tot, normalise_data, transform_data, generate_data, create_model 
from toolbox.plots.Plot import plot_history, plot_results

## Parameters to be set

Before running everything, here all parameters needed to run the code are listed, change them as you want

num_folds: Number of Cross-Validation subsets that the user'd
like to divide your subset in.

splits_to_train_and_test: list containing which of the splits the
user'd like to be trained and tested during this run, starting in 0
for the first split ant so on

normalize: parameter to define if the train and test will be performed
using normalized samples or not

cut_signals: parameter to define if the signals are already cut
in the size that the user wants the network to train on or if it's
necessary to be cut. The user should insert the size of the final desired vector (that is, seconds* sampling frequency)

bias_compensation: parameter to define which bias compensation method
is desired by the user

In [3]:
dataset_number=1

animal_number= 1

length_samples=50

classifier_chosen='EEGNet'
kernLength=100

# Nociception, dorsiflexion, plantarflexion, touch or 
# Nociception, dorsiflexion, plantarflexion, touch, resting
n_classes = 4

string_splits='1,2,3,4,5'

# Ring or standard (electrode 1-16) or Longitudinal (1, 5, 9, 13, 2, 6, 10, 14, 3, 7, 11, 15, 4, 8, 12, 16) or 
# Random
#flag_longitudinal='Longitudinal'
flag_longitudinal=''

In [4]:
#splits_to_train_and_test=np.empty(math.floor(len(string_splits)/2)+1)
num_folds = int(len(string_splits.split(',')))

test_size=0.2 # 80% train+val, 20% test 

bias_compensation=0

#bias_compensation=int(input('Which method would you like to use to balance your data? \n0: None \n1: Class weights \n2: Over-sampling \n3: Under-sampling \n'))

## Integrating GPU

If you have a GPU at hand and are able to integrate it to tensorflow, use it!

In [5]:
#Cell used to check if there are any GPUs integrated with tensorflow that can be used
print(tf.config.list_physical_devices('GPU'))

[]


## Fixed parameters
These parameters are never going to be changed when using Newcastle's Dataset, so it's not an input by the user. However, IF you desire to use the CNN with a different Dataset, please change these parameters or make them an input as the following above.

In [6]:
# Total of electrodes
n_electrodes = 16
n_features = n_electrodes

# After downsampling, the sampling frequency is Elisa's Pipeline was 5kHz, change if necessary
fs = 5000
ns = int(length_samples/1000*5000)  # samples

#max_epochs = 120
#start_epoch = 30
# pat value should be approx 10% of number of epochs
#pat_value=12

max_epochs = 3
start_epoch = 1
pat_value=1

ns 

250

# Importing data


In [7]:
dataset='dataset '+str(dataset_number)
animal='Animal '+str(animal_number)
length=str(length_samples)+'ms'

### Define from which folder the samples will be acquired

In this section, data will be imported from the folder that the user desires. It is important to set the path correctly to the folder containing all data that is needed for training.
For both paths, leave all bars like this '/'. path_animal is the output path for Classificator, animal and window

In [8]:
#path_folder='PATH/'+dataset+'/'+animal+'/'+length

### Define which folder do you want to save the outputs and log file for tensorboard

In [9]:
# Base folder where the script is executed
base_dir = os.getcwd()  # Get current working directory

# Construct path for the specific animal's results
path_animal = os.path.join(
    base_dir, 
    'results ' + dataset, 
    classifier_chosen, 
    f'{n_classes}_class', 
    flag_longitudinal, 
    animal
)

# Base path for logs (kept relative to script execution folder)
path_log = os.path.join(base_dir, 'logs')  # You can change 'logs' to desired folder

# Full output path including specific length subfolder
path_output = os.path.join(path_animal, length)

# Create the directories if they don't exist
os.makedirs(path_output, exist_ok=True)

# Display the created path
print(path_output)


C:\Users\david\DatiTesiRAW\Github\PyPNRelay\results dataset 1\EEGNet\4_class\Animal 1\50ms


## Preprocessing
This section is used to generate random samples if needed, or overlap.

## Qui vengono generati samples random (solo per vedere se è tutto ok con il training)

In [10]:
x_samp, y_samp = generate_data(20, n_electrodes, ns, n_classes)
print(x_samp.shape)
print(y_samp.shape)
#print(signals[5])
print(y_samp)

(80, 16, 250)
(80,)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3]


In [11]:
print(x_samp.shape)
print(y_samp.shape)

(80, 16, 250)
(80,)


# Extracting samples and features for the dataset
if overlap_usage>0:
    x_samp_overlap,y_samp_overlap,file_idx_overlap,sample_idx_overlap,original_samples_overlap,n_samples_overlap=transform_data(file_name_overlap,file_paths_overlap,n_features)
    x_samp_overlap.shape

In [12]:
sequence_size = (x_samp.shape[2], n_features)
sequence_size

(250, 16)

## Split in train+validation and test

In [13]:
X_trainval, X_test, y_trainval, y_test = train_test_split(x_samp, y_samp, test_size=test_size, random_state=42)

print(X_trainval.shape)
print(X_test.shape)
print(y_trainval)
print(y_test)

(64, 16, 250)
(16, 16, 250)
[3 3 2 2 0 3 0 2 1 3 2 2 0 1 2 3 0 2 2 0 3 1 2 0 3 0 0 1 0 3 0 1 3 2 2 3 0
 1 2 1 2 1 2 3 2 0 1 3 2 3 3 1 1 0 2 1 0 1 3 1 3 3 0 2]
[1 0 1 1 0 1 0 3 0 0 2 1 3 1 3 2]


In [14]:
# If you just want the count of one class elements y_samp, you can use:
count_zero_elements = np.count_nonzero(y_samp == 4)
print(count_zero_elements)

0


## Cross-validation

Creates matrices x and y, where x stores the data itself and y stores the target value - which of the stimulations was performed - in an encoded manner

In [15]:
# Generates the stratified fold
skf=StratifiedKFold(n_splits=num_folds, shuffle=True,random_state=42)
skf.get_n_splits(X_trainval,y_trainval)

5

# Training

Model creation, training with the desired data and images creation

In [16]:
# Initialize variables to store cross-validation results
cv_results = {'val_accuracy': [], 'val_f1_score': [], 'val_weighted_f1_score': [], 'val_best_weights': [], 'val_loss': [], 'train_acc_history': [], 'val_acc_history': [], 'val_confusion_matrix': []}
test_results = {'test_accuracy': [], 'test_f1_score': [],'test_weighted_f1_score': [], 'test_confusion_matrix': []}

In [17]:
for fold, (train_index, val_index) in enumerate(skf.split(X_trainval, y_trainval)):

    print(f"FOLD {fold+1}/{num_folds}") 
    
    # Create a unique folder for each CV fold
    cv_folder = f'CV{fold + 1}'
    output_folder_cv = path_output+'/'+cv_folder
    os.makedirs(output_folder_cv, exist_ok=True)
    
    # Create the model
    model = create_model(path_output, classifier_chosen, n_classes, ns, kernLength)
    print(model)
    # Compile the model
    opt = keras.optimizers.Adam(learning_rate=0.0002)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    # Set tensorboard
    log_dir = path_log+"/logs/fit/"+classifier_chosen+'/'+"Animal"+str(animal_number)+'/'+str(length_samples)+"ms"+'/'+cv_folder+'_'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
    
    # Separate each cv in visualization
    #writer.add_scalars('runs_split{}'.format(fold), {'Loss/train': training_loss, 'Loss/validation': validation_loss}, epoch+1)
    
    
####################################### TRAINING AND VALIDATION ##################################       
   
    X_fold_train, X_fold_val = X_trainval[train_index], X_trainval[val_index]
    y_fold_train, y_fold_val = y_trainval[train_index], y_trainval[val_index]
    
    print('train_index\n') 
    print(train_index)
    print('val_index\n')
    print(val_index)

    # Define EarlyStopping and ModelCheckpoint callbacks
    early_stopping = EarlyStopping(monitor='val_loss', patience=pat_value, min_delta=0.002, start_from_epoch=start_epoch,  verbose=1)
    checkpoint_path = os.path.join(output_folder_cv, 'best_model_checkpoint.h5')
    model_checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, verbose=1)

    # Train the model
    history = model.fit(X_fold_train, y_fold_train, epochs=max_epochs,
                        validation_data=(X_fold_val, y_fold_val),
                        callbacks=[early_stopping, model_checkpoint, tensorboard_callback],
                        verbose=1)
    
    # Load the best model weights
    model.load_weights(checkpoint_path)
    
    # Calculate metrics on validation data
    y_val_pred = model.predict(X_fold_val)
    y_val_pred_nohot = np.argmax(y_val_pred, axis=1)  # Convert probabilities to class labels
        
    print('y_val_pred\n') 
    print(y_val_pred)
    print('y_val_pred_nohot\n') 
    print(y_val_pred_nohot)

    # Calculate F1-scores for macro and weighted, confusion matrix
    f1 = f1_score(y_fold_val, y_val_pred_nohot, average='macro')
    weighted_f1 = f1_score(y_fold_val, y_val_pred_nohot, average='weighted')
    conf_matrix_val = confusion_matrix(y_fold_val, y_val_pred_nohot)

    # Store results in the dictionary
    cv_results['val_accuracy'].append(accuracy_score(y_fold_val, y_val_pred_nohot))
    cv_results['val_f1_score'].append(f1)
    cv_results['val_weighted_f1_score'].append(weighted_f1)
    cv_results['val_confusion_matrix'].append(conf_matrix_val)
    cv_results['val_best_weights'].append(model.get_weights())
    
    cv_results['val_loss'].append(history.history['loss'])
    cv_results['train_acc_history'].append(history.history['accuracy'])
    cv_results['val_acc_history'].append(history.history['val_accuracy'])
    
####################################### TESTING #######################################       

    y_test_pred = model.predict(X_test)
    y_test_pred_nohot = np.argmax(y_test_pred, axis=1)
    
    # Calculate testing metrics
    test_results['test_accuracy'].append(accuracy_score(y_test, y_test_pred_nohot))
    test_results['test_f1_score'].append(f1_score(y_test, y_test_pred_nohot, average='macro'))
    test_results['test_weighted_f1_score'].append(f1_score(y_test, y_test_pred_nohot, average='weighted'))
    test_results['test_confusion_matrix'].append(confusion_matrix(y_test, y_test_pred_nohot))
    
####################################### PLOT RESULTS #######################################

    plot_history(output_folder_cv, history, fold)
    
####################################### SAVE RESULTS #######################################       
    
    # Combine all the results into a single dictionary
    results= {
        'cv_results': cv_results,
        'test_results': test_results
    }

    output_file = path_output + '/' +  classifier_chosen + '_'+ str(n_classes)+'_class_'+ animal + '_' + length  + '_results.pickle'

    # Save the dictionary to a file using pickle
    with open(output_file, 'wb') as file:
        pickle.dump(results, file)

FOLD 1/5
train_index

[ 2  3  4  5  7  9 11 12 13 14 15 17 18 19 20 22 23 24 25 26 27 28 29 30
 31 33 35 36 37 38 39 40 41 42 43 44 46 47 48 49 50 51 53 54 55 56 57 58
 59 61 62]
val_index

[ 0  1  6  8 10 16 21 32 34 45 52 60 63]
Epoch 1/3
1/2 [==============>...............] - ETA: 1s - loss: 1.5992 - accuracy: 0.2500
Epoch 1: val_accuracy improved from -inf to 0.15385, saving model to C:\Users\david\DatiTesiRAW\Github\PyPNRelay\results dataset 1\EEGNet\4_class\Animal 1\50ms/CV1\best_model_checkpoint.h5
2/2 [==============================] - 2s 407ms/step - loss: 1.5462 - accuracy: 0.2549 - val_loss: 1.3865 - val_accuracy: 0.1538
Epoch 2/3
1/2 [==============>...............] - ETA: 0s - loss: 1.4263 - accuracy: 0.2188
Epoch 2: val_accuracy improved from 0.15385 to 0.30769, saving model to C:\Users\david\DatiTesiRAW\Github\PyPNRelay\results dataset 1\EEGNet\4_class\Animal 1\50ms/CV1\best_model_checkpoint.h5
2/2 [==============================] - 0s 192ms/step - loss: 1.4125 - accurac

1/1 [==============================] - 0s 23ms/step
FOLD 5/5
train_index

[ 0  1  3  5  6  7  8 10 11 12 13 14 15 16 17 18 19 21 23 24 25 27 28 29
 30 32 33 34 35 36 37 38 39 41 44 45 47 48 49 50 51 52 54 55 56 57 58 59
 60 61 62 63]
val_index

[ 2  4  9 20 22 26 31 40 42 43 46 53]
Epoch 1/3
1/2 [==============>...............] - ETA: 1s - loss: 1.6703 - accuracy: 0.1875WARNING:tensorflow:5 out of the last 13 calls to <function Model.make_test_function.<locals>.test_function at 0x000001E4CE27DB20> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and http

 ## Adaptation for boxplot file

In [18]:
accuracy_val_array = np.array(cv_results['val_accuracy'])
f1score_val_array = np.array(cv_results['val_f1_score'])

accuracy_test_array = np.array(test_results['test_accuracy'])
f1score_test_array = np.array(test_results['test_f1_score'])

np.savez(os.path.join(path_animal, f"{animal}_Validation_arrays_{length}.npz"), accuracy=accuracy_val_array, f1score=f1score_val_array)
np.savez(os.path.join(path_animal, f"{animal}_Test_arrays_{length}.npz"), accuracy=accuracy_test_array, f1score=f1score_test_array)


notebook.start("--logdir "+path_log)

# Per far ripartire Tensorboard
cancellare .tensorboard-info in temp folder e runnare con clear output

import tempfile
print(tempfile.gettempdir())

print(path_output+"logs/fit/")

In [19]:
#notebook.start("--logdir "+"C:/Users/david/DatiTesiRAW/EEGNet/V2_EEGNet/EEGNet_test")